# Packaging Python and Node Graphs for Production
Companion notebook for the **Packaging Python and Node Graphs for Production** section of Chapter 12.

The main rule: keep **code, config, and secrets separate**. This notebook works against the two real Dockerfiles shipped alongside it -- `Dockerfile.node` and `Dockerfile.python` -- rather than just displaying the text as a string.

In [1]:
print(open("Dockerfile.node").read())

FROM node:20 AS build
WORKDIR /app
COPY package*.json ./
RUN npm ci
COPY . .
RUN npm run build

FROM node:20-slim
WORKDIR /app
COPY package*.json ./
RUN npm ci --omit=dev
COPY --from=build /app/dist ./dist
CMD ["node", "dist/server.js"]



In [2]:
print(open("Dockerfile.python").read())

FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["python", "-m", "service.main"]



A small structural check for the packaging rule the book lists: **install dependencies first, copy application code after that**. Getting this order backwards means every code change invalidates Docker's dependency-install layer cache, turning a one-line fix into a full reinstall on every build:

In [3]:
def check_deps_before_code(dockerfile_text: str) -> bool:
    """For each build stage, the last COPY/RUN that installs dependencies
    (npm ci / pip install) should come before the COPY that pulls in full app code."""
    lines = [l.strip() for l in dockerfile_text.splitlines() if l.strip()]
    install_idx = next(
        (i for i, l in enumerate(lines) if "npm ci" in l or "pip install" in l), None
    )
    full_copy_idx = next(
        (i for i, l in enumerate(lines) if l.startswith("COPY") and l.strip() in {"COPY . .", "COPY ..."}), None
    )
    if install_idx is None or full_copy_idx is None:
        return False
    return install_idx < full_copy_idx


for name in ["Dockerfile.node", "Dockerfile.python"]:
    text = open(name).read()
    print(f"{name}: deps installed before full code copy = {check_deps_before_code(text)}")

Dockerfile.node: deps installed before full code copy = True
Dockerfile.python: deps installed before full code copy = True


**Repeatable builds**: pin exact versions instead of loose ranges, the same rule Chapter 7 covered for `requirements.in` / `requirements.txt` -- it applies equally to packaging for deployment. In Node, the equivalent is committing the lock file that matches your package manager:

In [4]:
lockfile_by_manager = {
    "npm": "package-lock.json",
    "yarn": "yarn.lock",
    "pnpm": "pnpm-lock.yaml",
}


def required_lockfile(package_manager: str) -> str:
    if package_manager not in lockfile_by_manager:
        raise ValueError(f"unknown package manager: {package_manager}")
    return lockfile_by_manager[package_manager]


print(required_lockfile("npm"))
print(required_lockfile("pnpm"))

package-lock.json
pnpm-lock.yaml


**Clean entry point**: one explicit startup command, injected config, small build output. Both Dockerfiles above end in exactly one `CMD` -- a service with two or three possible entry points is harder to health-check and harder to reason about in a process manager:

In [5]:
import re


def count_cmd_instructions(dockerfile_text: str) -> int:
    return len(re.findall(r"^CMD\s", dockerfile_text, flags=re.MULTILINE))


for name in ["Dockerfile.node", "Dockerfile.python"]:
    print(name, "-> CMD instructions:", count_cmd_instructions(open(name).read()))

Dockerfile.node -> CMD instructions: 1
Dockerfile.python -> CMD instructions: 1
